# Lab 09: Production-Grade App with Full LangFuse Observability

**Goal:** Build a complete production-ready FastAPI + LangGraph application with comprehensive LangFuse observability.

**What you'll learn:**
- Integrate LangFuse with the Session 11 production FastAPI app
- Add CallbackHandler to LangGraph agent endpoints
- Implement cost tracking per request
- Add user feedback collection endpoints
- Configure health probes with observability metrics
- Test the complete instrumented system

**Prerequisites:**
- Session 11: Production FastAPI + LangGraph app
- Session 12 Labs 01-08: LangFuse fundamentals

**Architecture:**
```
User Request
    ↓
FastAPI Endpoint (/api/support)
    ↓
LangGraph Agent (with CallbackHandler)
    ↓
LangFuse Mock (traces.json)
    ↓
Cost Tracking + Feedback + Health Metrics
```

## Setup

In [ ]:
import os
import shutil
import json
from datetime import datetime
from typing import TypedDict, Annotated, Optional
from operator import add

WORKDIR = "/tmp/prod-lab-12-09"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

print(f"Working directory: {WORKDIR}")

## Step 1: MockLangfuse Setup (from Session 12 Lab 02)

In [ ]:
class MockLangfuse:
    """Mock LangFuse client that logs traces to local JSON files."""

    def __init__(self, public_key, secret_key, host, output_dir="/tmp/langfuse-traces"):
        self.public_key = public_key
        self.secret_key = secret_key
        self.host = host
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self._traces = []

    def trace(self, name, user_id=None, session_id=None, metadata=None, tags=None):
        trace_data = {
            "id": f"trace-{len(self._traces)+1}",
            "name": name,
            "timestamp": datetime.now().isoformat(),
            "user_id": user_id,
            "session_id": session_id,
            "metadata": metadata or {},
            "tags": tags or [],
            "generations": [],
            "scores": [],
            "cost": 0.0,
        }
        self._traces.append(trace_data)
        return MockTrace(trace_data)

    def flush(self):
        output_file = os.path.join(self.output_dir, "production_traces.json")
        with open(output_file, "w") as f:
            json.dump(self._traces, f, indent=2)
        print(f"✓ Flushed {len(self._traces)} traces to {output_file}")
        return len(self._traces)

    def get_traces(self, user_id=None, session_id=None):
        traces = self._traces
        if user_id:
            traces = [t for t in traces if t.get("user_id") == user_id]
        if session_id:
            traces = [t for t in traces if t.get("session_id") == session_id]
        return traces

    def score(self, trace_id, name, value, data_type="NUMERIC", comment=None):
        """Add a score to a trace."""
        for trace in self._traces:
            if trace["id"] == trace_id:
                trace["scores"].append({
                    "name": name,
                    "value": value,
                    "data_type": data_type,
                    "comment": comment,
                    "timestamp": datetime.now().isoformat(),
                })
                return
        raise ValueError(f"Trace {trace_id} not found")


class MockTrace:
    def __init__(self, trace_data):
        self._data = trace_data
        self.id = trace_data["id"]

    def generation(self, name, model=None, input=None, output=None, 
                   usage=None, metadata=None):
        """Add a generation (LLM call) to this trace."""
        gen = {
            "name": name,
            "model": model,
            "input": input,
            "output": output,
            "usage": usage or {},
            "metadata": metadata or {},
            "timestamp": datetime.now().isoformat(),
        }
        
        # Calculate cost
        if usage and model:
            cost = self._calculate_cost(model, usage)
            gen["cost"] = cost
            self._data["cost"] += cost
        
        self._data["generations"].append(gen)
        return gen

    def _calculate_cost(self, model, usage):
        """Calculate cost based on model pricing."""
        PRICING = {
            "llama-3.3-70b-versatile": (0.00000059, 0.00000079),  # input, output per token
            "llama-3.2-1b": (0.00000004, 0.00000004),
            "mixtral-8x7b-32768": (0.00000027, 0.00000027),
        }
        
        input_price, output_price = PRICING.get(model, (0, 0))
        input_tokens = usage.get("input_tokens", 0)
        output_tokens = usage.get("output_tokens", 0)
        
        return (input_tokens * input_price) + (output_tokens * output_price)


# Initialize mock LangFuse client
langfuse = MockLangfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY", "pk-mock"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY", "sk-mock"),
    host=os.getenv("LANGFUSE_HOST", "http://localhost:8000"),
    output_dir=WORKDIR,
)

print("✓ MockLangfuse initialized")

## Step 2: Production LangGraph Agent (from Session 11)

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

load_dotenv()

# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Agent State
class SupportState(TypedDict):
    request: str
    employee_name: str
    category: str
    worker_output: str
    error: str
    final_response: str
    audit: Annotated[list, add]
    trace_id: Optional[str]


# Agent Nodes
TEMPLATES = {
    "hr": "Please visit the HR portal or email hr@company.com.",
    "tech": "Please create a Jira ticket or contact IT at ext. 5555.",
    "finance": "Please email finance@company.com with details.",
    "general": "Your request has been noted. A team member will respond shortly.",
}


def supervisor(state: SupportState) -> dict:
    """Classify the support request."""
    prompt = f"Classify as: hr, tech, finance, general. One word.\n{state['request']}"
    try:
        response = llm.invoke(prompt)
        cat = response.content.strip().lower()
        if cat not in ["hr", "tech", "finance", "general"]:
            cat = "general"
    except Exception:
        cat = "general"
    return {
        "category": cat,
        "error": "",
        "audit": [f"Supervisor: classified as {cat}"],
    }


def worker(state: SupportState) -> dict:
    """Generate response for the category."""
    prompt = (
        f"You are {state['category']} support.\n"
        f"Request: {state['request']}\n"
        f"Reply helpfully in 2 sentences."
    )
    try:
        response = llm.invoke(prompt)
        return {
            "worker_output": response.content.strip(),
            "error": "",
            "audit": [f"Worker ({state['category']}) responded"],
        }
    except Exception as e:
        return {
            "worker_output": TEMPLATES.get(state["category"], TEMPLATES["general"]),
            "error": str(e),
            "audit": [f"Worker error, used template"],
        }


def finalize(state: SupportState) -> dict:
    """Format final response."""
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {
        "final_response": f"[{state['category'].upper()}] {state['worker_output']}\n— UniGPS Support | {ts}",
        "audit": [f"Finalized at {ts}"],
    }


# Build LangGraph
graph = StateGraph(SupportState)
graph.add_node("supervisor", supervisor)
graph.add_node("worker", worker)
graph.add_node("finalize", finalize)
graph.add_edge(START, "supervisor")
graph.add_edge("supervisor", "worker")
graph.add_edge("worker", "finalize")
graph.add_edge("finalize", END)
agent = graph.compile()

print("✓ LangGraph agent compiled")

## Step 3: Production FastAPI with LangFuse Instrumentation

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
import asyncio
import time

app = FastAPI(
    title="UniGPS Support Agent API",
    version="1.0.0",
    description="Production-grade support agent with LangFuse observability",
)

# Request/Response models
class SupportRequest(BaseModel):
    employee_name: str
    request: str
    session_id: Optional[str] = None


class SupportResponse(BaseModel):
    category: str
    response: str
    audit: list[str]
    trace_id: str
    cost: float
    latency_ms: int


class FeedbackRequest(BaseModel):
    trace_id: str
    rating: int  # 1-5
    comment: Optional[str] = None


class HealthResponse(BaseModel):
    status: str
    version: str
    agent: str
    observability: str
    total_requests: int
    total_cost: float


# In-memory metrics
metrics = {
    "total_requests": 0,
    "total_cost": 0.0,
}


@app.get("/health", response_model=HealthResponse)
async def health():
    """Health probe with observability metrics."""
    return HealthResponse(
        status="healthy",
        version="1.0.0",
        agent="ready",
        observability="langfuse-mock",
        total_requests=metrics["total_requests"],
        total_cost=round(metrics["total_cost"], 4),
    )


@app.post("/api/support", response_model=SupportResponse)
async def handle_support(req: SupportRequest):
    """Handle support request with full LangFuse observability."""
    start_time = time.time()
    
    # Create LangFuse trace
    session_id = req.session_id or f"session-{int(time.time())}"
    trace = langfuse.trace(
        name="support_request",
        user_id=req.employee_name,
        session_id=session_id,
        metadata={
            "endpoint": "/api/support",
            "request": req.request[:100],  # First 100 chars
        },
        tags=["production", "support"],
    )
    
    # Invoke LangGraph agent
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        None,
        lambda: agent.invoke({
            "request": req.request,
            "employee_name": req.employee_name,
            "category": "",
            "worker_output": "",
            "error": "",
            "final_response": "",
            "audit": [],
            "trace_id": trace.id,
        })
    )
    
    # Log generations to LangFuse
    # Note: In real implementation, this would be done via CallbackHandler
    # Here we simulate with manual logging
    trace.generation(
        name="supervisor_classify",
        model="llama-3.3-70b-versatile",
        input=f"Classify: {req.request}",
        output=result["category"],
        usage={"input_tokens": 50, "output_tokens": 5},
        metadata={"step": "supervisor"},
    )
    
    trace.generation(
        name="worker_respond",
        model="llama-3.3-70b-versatile",
        input=f"{result['category']} support: {req.request}",
        output=result["worker_output"],
        usage={"input_tokens": 150, "output_tokens": 80},
        metadata={"step": "worker", "category": result["category"]},
    )
    
    # Calculate latency
    latency_ms = int((time.time() - start_time) * 1000)
    
    # Update metrics
    metrics["total_requests"] += 1
    metrics["total_cost"] += langfuse.get_traces()[-1]["cost"]
    
    # Flush traces
    langfuse.flush()
    
    return SupportResponse(
        category=result["category"],
        response=result["final_response"],
        audit=result["audit"],
        trace_id=trace.id,
        cost=round(langfuse.get_traces()[-1]["cost"], 6),
        latency_ms=latency_ms,
    )


@app.post("/api/feedback")
async def submit_feedback(feedback: FeedbackRequest):
    """Submit user feedback for a trace."""
    try:
        langfuse.score(
            trace_id=feedback.trace_id,
            name="user_rating",
            value=feedback.rating,
            data_type="NUMERIC",
            comment=feedback.comment,
        )
        langfuse.flush()
        return {"status": "success", "trace_id": feedback.trace_id}
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))


@app.get("/api/traces/{user_id}")
async def get_user_traces(user_id: str):
    """Get all traces for a specific user."""
    traces = langfuse.get_traces(user_id=user_id)
    return {
        "user_id": user_id,
        "total_traces": len(traces),
        "total_cost": sum(t["cost"] for t in traces),
        "traces": traces,
    }


print("✓ FastAPI app configured with LangFuse instrumentation")

## Step 4: Test the Production System

In [ ]:
client = TestClient(app)

# Test 1: Health check
print("=== Test 1: Health Check ===")
resp = client.get("/health")
health_data = resp.json()
print(f"Status: {health_data['status']}")
print(f"Agent: {health_data['agent']}")
print(f"Observability: {health_data['observability']}")
print(f"Total requests: {health_data['total_requests']}")
print()

In [ ]:
# Test 2: Support request with observability
print("=== Test 2: Support Request ===")
resp = client.post("/api/support", json={
    "employee_name": "Priya",
    "request": "I need to apply for sick leave",
    "session_id": "test-session-001",
})

data = resp.json()
print(f"Category: {data['category']}")
print(f"Response: {data['response'][:80]}...")
print(f"Trace ID: {data['trace_id']}")
print(f"Cost: ${data['cost']:.6f}")
print(f"Latency: {data['latency_ms']}ms")
print(f"Audit trail: {data['audit']}")
print()

trace_id_1 = data['trace_id']

In [ ]:
# Test 3: Another support request (different category)
print("=== Test 3: Tech Support Request ===")
resp = client.post("/api/support", json={
    "employee_name": "Vikram",
    "request": "My VPN keeps disconnecting",
    "session_id": "test-session-002",
})

data = resp.json()
print(f"Category: {data['category']}")
print(f"Response: {data['response'][:80]}...")
print(f"Trace ID: {data['trace_id']}")
print(f"Cost: ${data['cost']:.6f}")
print(f"Latency: {data['latency_ms']}ms")
print()

trace_id_2 = data['trace_id']

In [ ]:
# Test 4: Submit user feedback
print("=== Test 4: User Feedback ===")
resp = client.post("/api/feedback", json={
    "trace_id": trace_id_1,
    "rating": 5,
    "comment": "Very helpful response!",
})

print(f"Feedback submitted: {resp.json()}")
print()

resp = client.post("/api/feedback", json={
    "trace_id": trace_id_2,
    "rating": 4,
    "comment": "Good, but could be more specific.",
})

print(f"Feedback submitted: {resp.json()}")
print()

In [ ]:
# Test 5: Get user traces
print("=== Test 5: User Traces ===")
resp = client.get("/api/traces/Priya")
user_data = resp.json()
print(f"User: {user_data['user_id']}")
print(f"Total traces: {user_data['total_traces']}")
print(f"Total cost: ${user_data['total_cost']:.6f}")
print()

if user_data['traces']:
    trace = user_data['traces'][0]
    print(f"Sample trace:")
    print(f"  ID: {trace['id']}")
    print(f"  Session: {trace['session_id']}")
    print(f"  Generations: {len(trace['generations'])}")
    print(f"  Scores: {len(trace['scores'])}")
    if trace['scores']:
        print(f"  User rating: {trace['scores'][0]['value']}/5")

In [ ]:
# Test 6: Final health check
print("\n=== Test 6: Final Health Check ===")
resp = client.get("/health")
health_data = resp.json()
print(f"Status: {health_data['status']}")
print(f"Total requests: {health_data['total_requests']}")
print(f"Total cost: ${health_data['total_cost']:.6f}")
print()

## Step 5: View Trace Data

In [ ]:
# Load and display trace file
trace_file = os.path.join(WORKDIR, "production_traces.json")
with open(trace_file) as f:
    all_traces = json.load(f)

print(f"=== LangFuse Trace Summary ===")
print(f"Total traces: {len(all_traces)}")
print(f"Trace file: {trace_file}")
print()

for i, trace in enumerate(all_traces, 1):
    print(f"Trace {i}:")
    print(f"  ID: {trace['id']}")
    print(f"  User: {trace['user_id']}")
    print(f"  Session: {trace['session_id']}")
    print(f"  Tags: {trace['tags']}")
    print(f"  Generations: {len(trace['generations'])}")
    print(f"  Cost: ${trace['cost']:.6f}")
    print(f"  Scores: {len(trace['scores'])}")
    if trace['scores']:
        for score in trace['scores']:
            print(f"    - {score['name']}: {score['value']} ({score.get('comment', 'no comment')})")
    print()

In [ ]:
# Display detailed trace structure
print("=== Detailed Trace Example ===")
sample_trace = all_traces[0]
print(json.dumps(sample_trace, indent=2))

## TODO 1: Add Cost Analysis

Analyze the trace data and calculate:
- Total cost across all traces
- Cost per user
- Average cost per request
- Most expensive trace
- Cost by category (HR, Tech, Finance, General)

In [ ]:
# TODO: Cost analysis
# Your code here

total_cost = 0.0  # Calculate from all_traces
cost_per_user = {}  # Dict[user_id, total_cost]
avg_cost_per_request = 0.0
most_expensive_trace = None  # Find trace with highest cost

print("=== Cost Analysis ===")
print(f"Total cost: ${total_cost:.6f}")
print(f"Average cost per request: ${avg_cost_per_request:.6f}")
print(f"Cost per user: {cost_per_user}")
if most_expensive_trace:
    print(f"Most expensive trace: {most_expensive_trace['id']} (${most_expensive_trace['cost']:.6f})")

## TODO 2: Add Quality Metrics

Calculate quality metrics from user feedback:
- Average user rating
- Rating distribution (count by rating value)
- Percentage of traces with feedback
- Identify traces with low ratings (< 3)

In [ ]:
# TODO: Quality metrics
# Your code here

avg_rating = 0.0  # Calculate from scores
rating_distribution = {}  # Dict[rating, count]
feedback_percentage = 0.0  # Percentage of traces with scores
low_rated_traces = []  # Traces with rating < 3

print("=== Quality Metrics ===")
print(f"Average user rating: {avg_rating:.2f}/5")
print(f"Rating distribution: {rating_distribution}")
print(f"Feedback coverage: {feedback_percentage:.1f}%")
print(f"Low-rated traces: {len(low_rated_traces)}")

## TODO 3: Production Checklist

Verify that all production requirements are met:

**Observability:**
- [ ] LangFuse traces capture user_id and session_id
- [ ] All LLM calls are logged as generations
- [ ] Cost is calculated and tracked per request
- [ ] User feedback can be submitted and linked to traces

**API Design:**
- [ ] Health probe returns observability metrics
- [ ] Support endpoint returns trace_id to client
- [ ] Feedback endpoint validates trace_id
- [ ] Traces endpoint allows user-specific filtering

**Performance:**
- [ ] Async handling for agent invocation
- [ ] Latency tracking in response
- [ ] Traces flushed after each request

**Error Handling:**
- [ ] Agent errors logged in audit trail
- [ ] Fallback templates for worker failures
- [ ] 404 error for invalid trace_id in feedback

Fill in the checklist by running the tests above and verifying each requirement.

## Summary

This lab integrated:

**From Session 11 (Production):**
- FastAPI application with async handling
- LangGraph multi-agent system
- Health probes and structured responses
- Production-grade error handling

**From Session 12 (LangFuse):**
- MockLangfuse client with trace/generation API
- Trace creation with user_id, session_id, tags
- Generation logging with usage and cost
- User feedback collection via scores
- Cost tracking and analysis

**End Result:**
✅ Production-ready FastAPI + LangGraph support agent
✅ Full LangFuse observability (traces, generations, scores)
✅ Cost tracking per request and per user
✅ User feedback collection and analysis
✅ Health probes with observability metrics
✅ Complete audit trail for debugging

This is the foundation for a production AI agent system with comprehensive observability!

## Key Takeaways

1. **LangFuse Integration**: Adding observability requires creating traces, logging generations, and flushing after each request
2. **Cost Tracking**: Calculate cost per generation using model pricing and token usage
3. **User Feedback**: Link scores to traces via trace_id for quality monitoring
4. **Health Metrics**: Expose observability metrics in health probes for monitoring
5. **Production Pattern**: Trace creation → Agent invocation → Generation logging → Flush → Return trace_id
6. **Mock Mode**: Same patterns work with MockLangfuse (local JSON) and real LangFuse (PostgreSQL backend)

**Next Steps:**
- For real production: Replace MockLangfuse with actual LangFuse server
- Add LangChain CallbackHandler for automatic trace generation
- Set up LangFuse dashboard views for cost/latency/feedback analysis
- Configure alerts for cost spikes and low ratings
- Implement prompt management with `langfuse.get_prompt()`